# Iteration 3 — Melancholic Poet QLoRA Training (A/B experiment)

Two experiments in one run:
- **Experiment A (poet100):** 100 new strongly poetic examples only, 8 epochs
- **Experiment B (mixed250):** 250 examples (100 new + 150 old high/medium), 8 epochs

Both use LR 5e-5. Compare the adapters to see if pure signal beats diverse data.

**Runtime → Change runtime type → T4 GPU** before running.

## 1. Clone repo and install dependencies

In [ ]:
%cd /content
!rm -rf /content/PET_QLORA_POET
!git clone https://github.com/thejesusyung/melancholic-poet-qlora.git /content/PET_QLORA_POET

In [ ]:
%cd /content/PET_QLORA_POET/melancholic_poet_qlora
!pip install -r requirements.txt -q

import os, sys
os.environ["PYTHONPATH"] = "/content/PET_QLORA_POET/melancholic_poet_qlora/src"
sys.path.insert(0, "/content/PET_QLORA_POET/melancholic_poet_qlora/src")

## 2. Prepare training data (both experiments)

In [ ]:
# Experiment B data: all high+medium (250 train)
!python scripts/prepare_root_data.py \
  --source_dir /content/PET_QLORA_POET/data \
  --min_persona_strength medium

# Experiment A data: only the 100 new strongly poetic examples (json7.json)
!python scripts/prepare_root_data.py \
  --source_dir /content/PET_QLORA_POET/data \
  --source_pattern "json7.json" \
  --train_out data/generated/poet100_train.jsonl \
  --val_out data/generated/poet100_val.jsonl \
  --manifest_out data/generated/poet100_manifest.json

## 3. Experiment A: Train on 100 new poetic examples only

In [ ]:
!PYTHONPATH=/content/PET_QLORA_POET/melancholic_poet_qlora/src python train.py --config configs/poet100_qwen25_15b.yaml

## 4. Experiment B: Train on 250 mixed examples (100 new + 150 old high/medium)

In [ ]:
!PYTHONPATH=/content/PET_QLORA_POET/melancholic_poet_qlora/src python train.py --config configs/persona_only_qwen25_15b.yaml

## 5. Upload both adapters to S3

- **persona_only** slot → Experiment A (100 poetic examples)
- **mixed** slot → Experiment B (250 mixed examples)

In [ ]:
!pip install awscli -q

import os
os.environ["AWS_ACCESS_KEY_ID"] = ""       # fill in
os.environ["AWS_SECRET_ACCESS_KEY"] = ""   # fill in
os.environ["AWS_DEFAULT_REGION"] = "us-east-2"

In [ ]:
S3_BUCKET = "melancholic-poet-qlora-901059153423-us-east-2-an"

# Experiment A (100 only) → persona_only slot
!aws s3 sync outputs/poet100_qwen25_15b/adapter/ s3://{S3_BUCKET}/adapters/persona_only/ --delete

# Experiment B (250 mixed) → mixed slot
!aws s3 sync outputs/persona_only_qwen25_15b/adapter/ s3://{S3_BUCKET}/adapters/mixed/ --delete

print("Done. Restart the EC2 Gradio app to load the new adapters.")